# Reversal × volatility source preflight

Source and mechanism novelty only; see frozen `plan.json`.

In [1]:
import hashlib,json,os
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd();card=root/'research/stage1_reversal_vol_compounding_source_20261112';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
provider=root/plan['source_provider']
assert sha(provider/'qlib_store_manifest.json')==plan['source_manifest_sha256']
qlib.init(provider_uri=str(provider),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2017-01-01',end_time='2020-12-31',freq='day'))
raw=D.features(['SOXX','SOXL'],['$close'],start_time='2017-01-01',end_time='2020-12-31',freq='day')
close=raw['$close'].unstack('instrument').reindex(calendar)
assert {'SOXX','SOXL'}<=set(close)
soxx=close['SOXX'].astype(float);soxl=close['SOXL'].astype(float)
r=soxx.pct_change(fill_method=None);logr=np.log1p(r)
ac20=r.rolling(20,min_periods=20).corr(r.shift(1))
vol20_squared=r.rolling(20,min_periods=20).var(ddof=1)
path_eff=logr.rolling(21,min_periods=21).sum()/logr.abs().rolling(21,min_periods=21).sum()
soxl_log=np.log(soxl/soxl.shift(1));tracking=soxl_log-np.log1p(3*r)
product_vol=tracking.rolling(21,min_periods=21).std(ddof=1)
source=pd.DataFrame({'ac20':ac20,'vol20_squared':vol20_squared,'SOXX_return20':soxx/soxx.shift(20)-1,
 'signed_path_efficiency21':path_eff,'SOXL_tracking_error_vol21':product_vol})
source['reversal_vol']=np.maximum(-source.ac20,0)*source.vol20_squared
# All features at decision d come from the previous completed native session.
decision=source.shift(1).loc['2018-10-12':'2020-12-31'].copy();decision.index.name='decision_day'
controls=plan['controls'];known=decision[controls+['reversal_vol']].notna().all(axis=1)
use=decision.loc[known].copy();assert np.isfinite(use.to_numpy(float)).all()
annual={str(y):{'native_days':int((decision.index.year==y).sum()),'eligible_days':int((use.index.year==y).sum()),'positive_factor_days':int((use.loc[use.index.year==y,'reversal_vol']>0).sum())} for y in (2018,2019,2020)}
train=use.loc[use.index.year.isin([2019,2020])]
spearman={c:float(train[['reversal_vol',c]].corr(method='spearman').iloc[0,1]) for c in controls}
X=np.column_stack([np.ones(len(train)),train[controls].to_numpy(float)]);y=train.reversal_vol.to_numpy(float)
beta=np.linalg.lstsq(X,y,rcond=None)[0];resid=y-X@beta
residual_fraction=float(np.var(resid)/np.var(y))
source_gate=all(annual[str(y)]['eligible_days']>=200 and annual[str(y)]['positive_factor_days']>=20 for y in (2019,2020))
novelty_gate=all(abs(v)<.95 for v in spearman.values()) and residual_fraction>=.05
report={'decision':'source_novelty_pass_needs_information' if source_gate and novelty_gate else 'fixed_source_or_novelty_gate_failed',
 'historical_research_passed':False,'new_model_fits':0,'new_native_accounts':0,
 'provider_manifest_sha256':plan['source_manifest_sha256'],'qlib_calendar_rows':len(calendar),
 'raw_close_rows':len(raw),'annual':annual,'source_gate':source_gate,'spearman_to_controls':spearman,
 'residual_variance_fraction':residual_fraction,'novelty_gate':novelty_gate,
 'scope':'prior-session source only; no target label, account, alpha or independent forward evidence'}
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 R.save_objects(frozen_plan=plan,source_frame=use,source_audit=report)
 rec=R.get_recorder();rid,eid=rec.id,str(rec.experiment_id)
local=card/'qlib_recorder'/eid/rid;names=['frozen_plan','source_frame','source_audit']
manifest={'recorder_id':rid,'entries':[{'path':(local/'artifacts'/k).relative_to(card).as_posix(),'sha256':sha(local/'artifacts'/k),'bytes':(local/'artifacts'/k).stat().st_size} for k in names]}
(card/'verification_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
summary={'experiment':plan['experiment'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str((card/'qlib_recorder').resolve()),'recorder_local_dir':str(local.resolve()),'plan_sha256':sha(card/'plan.json'),**report,'artifact_policy':{'policy':'slim_card_recorder_first','recorder_only_artifacts':names}}
(card/'summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False)+'\n')
check,rr=load_verified_recorder_objects(card,expected_recorder_id=rid,artifacts=['source_frame','source_audit'],expected_manifest_sha256=sha(card/'verification_manifest.json'))
pd.testing.assert_frame_equal(check['source_frame'],use);assert check['source_audit']==report
(card/'readback_receipt.json').write_text(json.dumps({'recorder_id':rid,'summary_sha256':sha(card/'summary.json'),'manifest_sha256':sha(card/'verification_manifest.json'),'readback':rr},indent=2)+'\n')
print(json.dumps({'decision':report['decision'],'annual':annual,'residual_fraction':residual_fraction,'spearman':spearman,'recorder_id':rid},indent=2))

{
  "decision": "fixed_source_or_novelty_gate_failed",
  "annual": {
    "2018": {
      "native_days": 54,
      "eligible_days": 54,
      "positive_factor_days": 38
    },
    "2019": {
      "native_days": 252,
      "eligible_days": 252,
      "positive_factor_days": 181
    },
    "2020": {
      "native_days": 253,
      "eligible_days": 253,
      "positive_factor_days": 215
    }
  },
  "residual_fraction": 0.014608439561832304,
  "spearman": {
    "ac20": -0.9045386228807195,
    "vol20_squared": 0.6095424402939407,
    "SOXX_return20": 0.02416547326960256,
    "signed_path_efficiency21": -0.20788180678198576,
    "SOXL_tracking_error_vol21": 0.437849985009233
  },
  "recorder_id": "9efed5ef1e5840beaa02f3a0e2ca6d3b"
}


## Conclusion

**No SOXX pass.** 2019/2020 source support passed at 252/253 days. Residual source variance beyond the frozen controls was 1.46%, below the preregistered 5% novelty gate. This fixed formula closes before value labels, model fits or accounts. The Qlib Recorder and NAS scoped readback passed; see `summary.json`, `conclusion.md`, and receipts. This source test does not establish that all autocorrelation mechanisms are useless.